# Use Sentence Transformers Instead of word2vec

In [8]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import tensorflow as tf
import re

SEED = 222
pd.options.display.max_rows = 200
pd.options.display.min_rows = 30
pd.options.display.max_seq_items = None
np.random.seed(222)

In [3]:
# link drive
from google.colab import drive
drive.mount('/content/drive')
share_path = "/content/drive/MyDrive/Colab_Notebooks/deeplearning2026"
import os
print(os.listdir(share_path))

Mounted at /content/drive
['documentation', 'notebooks', 'data', 'NLP-Arch.gdoc', 'Siavash']


In [7]:
# load data
data_path = share_path + '/data/'
df = pd.read_parquet(data_path + 'FDCeverything.parquet.gzip')
df


nameunit,description,ALA_G,ARA_G,B12_UG,B1_MG,B2_MG,B3_MG,B5_MG,B6_MG,B7_UG,...,total_vit_d_UG,total_vit_e_mg,trans_fat_G,tryptophan_G,tyrosine_G,valine_G,vit_a_IU,vit_c_MG,vit_k_UG,water_G
0,advancepierre flamebroiled rib shaped pork pa...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.090,NaN,NaN,NaN,NaN,1.8,NaN,NaN
1,all natural gluten free chicken nuggets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,0.0,1.4,NaN,NaN
2,all natural rosemary & olive oil basmati rice...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,almond milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,208.0,0.0,NaN,NaN
4,"almond milk, original",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,208.0,0.0,NaN,NaN
5,artisan smooth & creamy gelato,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,NaN,0.9,NaN,NaN
6,"artisan smooth & creamy gelato, toasted coconut",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,NaN,0.9,NaN,NaN
7,artisanal collection spaghetti pasta,NaN,NaN,NaN,1.000,0.357,8.929,NaN,NaN,NaN,...,0.0,NaN,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,authentic barrel ripened feta cheese,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,barbecue sauce,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000,NaN,NaN,NaN,0.0,0.0,NaN,NaN


In [13]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[^\w\s,.-]", "", t)
    t = t.replace("%", " percent")
    t = t.replace("oz.", " ounce")
    t = t.replace("lb.", " pound")
    t = t.replace("&", "and")
    t = t.replace("w/", "with")
    t = t.replace("w/o", "without")
    t = t.replace("pkg", "package")
    return t.strip()


In [15]:
df["text"] = df['description'].apply(clean_text)

In [18]:
# inspect some results
print(df["text"].sample(5).tolist())

['hearts of palm lasagna sheets', 'cinnamon graham cookies, cinnamon', 'tequila red jalapeno hot sauce, tequila red jalapeno', 'buddig, smoked, chopped, pressed turkey, original', 'enriched soft bulkies rolls']


In [19]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    df["text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1904 [00:00<?, ?it/s]

In [21]:
embeddings.shape

(487227, 384)

In [23]:
df['embeddings'] = list(embeddings)

In [27]:
df.to_parquet(data_path + 'FDCwithTransformerEmbeddings.parquet.gzip',
              compression='gzip')